# Task 3 — kiểm tra chỉnh sửa ảnh bằng scene graph

Demo CPU từ **kết quả thực nghiệm thật**: 100 ảnh nguồn, 200 ảnh Qwen và 300 prediction REACT++. Chọn một case, xem ảnh/bbox/quan hệ có hướng, tính lại E/P/G và đối chiếu CLIPScore, LPIPS, nhãn tay.

- Model thị giác và CLIP/LPIPS **không chạy lại** trong notebook. Ảnh/graph/metric được nạp từ artifacts có SHA256.
- Điểm graph được **tính lại trên CPU** bằng scorer đã khóa. Nhãn tay chỉ dùng đối chiếu.
- Cuối notebook có phần riêng minh họa **dự đoán thật của Task 2 trên EMNLP**. Đây chưa phải pipeline Task 2 → Task 3 trên cùng một ảnh.

Trong workspace: chạy `bash scripts/package_level3.sh prepare` một lần trước khi mở. Sau khi giải nén bundle: dùng ngay manifest ở thư mục gốc. Cài `requirements-level3-demo-v1.txt` nếu cần.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "tools/level3_submission_v1/demo.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, HTML, Markdown  # noqa: E402
from tools.level3_submission_v1.demo import (  # noqa: E402 — after portable root setup
    open_bundle, recompute, verify_all_scores, image_pair_html, graph_html, table, task2_demo
)
root, manifest = open_bundle(ROOT)
print("Logical cohort:", manifest["counts"])
print("Nguồn ảnh:", {x: sum(s["dataset"] == x for s in manifest["sources"].values())
                       for x in ["sidewalk", "travel"]})

## 1. Chọn một yêu cầu

Mặc định là ca #58, thêm thuyền buồm. Các ví dụ khác:
- `travel-5335a75704bb20-q200-2`: thêm mèo, graph đánh giá bảo toàn quá cao.
- `travel-04ea5302b09c26-q200-2`: có chim nhưng tâm bbox nằm ngoài ROI định trước.
- Chọn một ID trong danh sách màu in bên dưới để thấy E/G không xác định.

Chỉ đổi `TASK_ID` rồi chạy lại các ô phía dưới. Không sửa manifest hoặc nhãn.

In [ ]:
TASK_ID = "travel-7bb0c6b3635205-q100-1"
SHOW_BOXES = True
print("Ví dụ đổi màu:", next(c["task_id"] for c in manifest["cases"]
                                if c["task"]["operation"]["kind"] == "attribute"))
case = recompute(root, manifest, TASK_ID)
display(Markdown("**Instruction:** " + case["instruction"]))
display(HTML(image_pair_html(root, case, boxes=SHOW_BOXES)))

## 2. Graph nguồn và graph đích

Mũi tên thể hiện chiều subject → object; bảng bên dưới ghi nhãn từng cạnh. Node ID chỉ có nghĩa trong từng graph, không được xem là ID đối tượng ổn định giữa hai ảnh. Bbox đỏ là detection; vùng vàng là ROI nguồn đã định trước. REACT++ hiện không có thuộc tính màu.

In [ ]:
left = graph_html(case["graphs_loaded"]["source"], "Graph nguồn", "source-arrow")
right = graph_html(case["graphs_loaded"]["target"], "Graph đích", "target-arrow")
display(HTML('<div style="display:flex;gap:24px;flex-wrap:wrap">'
             + '<div style="flex:1;min-width:300px">'+left+'</div>'
             + '<div style="flex:1;min-width:300px">'+right+'</div></div>'))
for role in ["source", "target"]:
    nodes = case["graphs_loaded"][role]["grounding"]["nodes"]
    display(Markdown("**Detection confidence — " + role + "** (chưa hiệu chuẩn trên bộ này)"))
    display(HTML(table(["ID", "Nhãn", "Confidence"],
                       [[k, v["label"], round(v["confidence"], 4) if v["confidence"] is not None else None]
                        for k,v in nodes.items()])))

## 3. Điểm graph được tính lại và nhãn tay

Ghép một-một theo nhãn + IoU ≥ 0,3. E kiểm tra các điều kiện thêm/xóa/thay trong vùng định trước; với replace, E là trung bình hai điều kiện. P là trung bình Dice node/cạnh bảo toàn có bằng chứng. G = E × P khi không có lý do abstain.

`None` là không chấm được, **không phải 0**. P vẫn có thể được lưu để chẩn đoán khi phép ghép mơ hồ; xem cột “P dùng được”. Các giá trị này không phải xác suất ảnh đúng hoặc F1 so với graph GT.

In [ ]:
s = case["recomputed"]
r = case["scores_and_labels"]
display(HTML(table(["E", "P", "P dùng được", "G", "CLIPScore", "LPIPS"],
                   [[s["edit_score"], s["preservation_score"], s["preservation_evaluable"],
                     s["graph_consistency"], r["clipscore"], r["lpips"]]])))
display(HTML(table(["Nhãn người duyệt: đúng yêu cầu", "Bảo toàn", "Đạt cả hai"],
                   [[r["edit_fulfillment"], r["preservation"], r["overall_pass"]]])))
print("Trạng thái:", s["status"], "| Lý do:", s["reason_codes"])
print("Điều kiện edit:", s.get("checks", []))
print("Dice node:", s["node_preservation"])
print("Dice cạnh:", s["edge_preservation"])
display(HTML(table(["Source ID", "Target ID", "IoU", "Mơ hồ"],
                   [[a["source"], a["target"], round(a["iou"], 4), a["ambiguous"]]
                    for a in s["alignment"]])))

## 4. Tái kiểm tra toàn bộ 200 điểm graph và bảng tương quan

Ô này tính lại scorer trên CPU cho đủ 200 cặp và kiểm tra khớp artifacts đã khóa. CLIPScore/LPIPS được đọc từ lượt inference GPU đã hoàn tất, không tính lại model. So sánh chính dùng đúng giao 91 cặp có đủ điểm. Trường nhãn trống được hiểu là đạt theo chỉ dẫn đã xác nhận của người duyệt.

In [ ]:
audit = verify_all_scores(root, manifest)
assert audit["pairs_recomputed"] == 200 and audit["common_n"] == 91
print(audit)
display(HTML(table(["Metric", "Spearman trên cùng 91 cặp"],
                   list(audit["spearman"].items()))))
print("Coverage G:", audit["graph_available"] / audit["pairs_recomputed"])
print("Chưa có bằng chứng graph vượt baseline ảnh. P với nhãn bảo toàn: rho ≈ 0.030.")

## 5. Thao tác graph dự đoán thật của Task 2 — minh họa riêng

Replay GNN/seed42, checkpoint có SHA trong output, trên một ca đúng và một ca sai thuộc bộ 10 ca EMNLP đã lưu. Áp dụng **predicted operations** vào source để dựng predicted target; target tham chiếu chỉ dùng đối chiếu. Đây không phải thao tác viết tay, không phải diff từ cặp ảnh Qwen và không chạy inference mới.

Graph EMNLP có thuộc tính dưới dạng node và link vô hướng được mã hóa thành cung đối xứng. Vì chưa có adapter được kiểm chứng sang graph REACT++, không gắn kết quả này với điểm G của case Task 3 ở trên.

In [ ]:
import json
for example in task2_demo(root, manifest):
    display(Markdown(f"### {example['id']} — exact theo evaluator đã lưu: {example['correct']}"))
    print("Instruction:", example["instruction"])
    print("Checkpoint SHA256:", example["checkpoint_sha256"])
    print("Predicted operations:")
    print(json.dumps(example["predicted"], ensure_ascii=False, indent=2))
    graphs = [("Nguồn Task 2", example["source"]),
              ("Đích dựng từ dự đoán", example["predicted_target"]),
              ("Đích tham chiếu", example["target"])]
    blocks = [f'<div style="flex:1;min-width:280px">{graph_html(g,title,"t2-"+str(i))}</div>'
              for i,(title,g) in enumerate(graphs)]
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:15px">'+''.join(blocks)+'</div>'))

## 6. Những điểm cần giải thích khi vấn đáp

1. Scene graph là dự đoán, nên thiếu detection không chứng minh vật thể thật đã bị xóa.
2. Câu yêu cầu được chuyển thành loại thao tác/anchor/ROI do assistant chuẩn bị từ nguồn; bộ chấm chưa tự hiểu mọi ràng buộc ngôn ngữ.
3. Graph không biểu diễn màu/kết cấu, và matching nhãn + bbox có thể không ổn định. Không dùng số node/edge như thước đo chất lượng.
4. CLIP center crop có thể cắt mất vật thể vừa thêm; LPIPS thấp chỉ cho biết ảnh ít thay đổi.
5. Task 2 và Task 3 đã có kết quả riêng; notebook không tuyên bố đã nối mô hình Task 2 vào toàn bộ 200 cặp REACT++.

Báo cáo: `docs/LEVEL3_TASK3_REPORT.md`. Data card: `docs/LEVEL3_DATA_CARD.md`. Giao diện 10 ca: đường dẫn `analysis_html` trong manifest. Tái lập ảnh/SGG dùng hai launcher độc lập trong `docs/LEVEL3_PIPELINE.md`.